In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from torch.utils.data import Dataset, DataLoader


In [7]:
# Configuration and Feature Definitions

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") #Automatically selects GPU acceleration (cuda) if available; otherwise falls back to cpu
BATCH_SIZE = 256 #Feeds data into the model in mini-batches of 256 samples.
EPOCHS = 100 # Defines the maximum training iterations over the dataset
PATIENCE = 15 # Sets early stopping threshold to stop training if validation performance stops improving for 15 consecutive epochs.
LEARNING_RATE = 1e-3 #Controls the optimizer step size ($0.001$)

#Specifies 17 numeric features (vehicle mass m (kg), engine capacity ec (cm3), power ep (KW), one-hot encoded fuel types, 
#and innovative technologies IT_*)
SELECTED_FEATURES = [
    "m (kg)",
    "ec (cm3)",
    "ep (KW)",
    "Ft_diesel",
    "Ft_petrol",
    "Ft_petrol/electric",
    "Ft_diesel/electric",
    "Fm_H",
    "Fm_M",
    "Fm_P",
    "Electric range (km)",
    "IT_28",
    "IT_29",
    "IT_32",
    "IT_33",
    "IT_37",
    "IT_38",
    "IT_none",
]

In [8]:
# PyTorch Dataset Definition

class VehicleDataset(Dataset):
    def __init__(self, X_df: pd.DataFrame, y_df: pd.DataFrame):
        # Converts pandas DataFrames into 32-bit floating point PyTorch Tensors        
        self.X = torch.tensor(X_df[SELECTED_FEATURES].values, dtype=torch.float32)
        # Reshapes targets to a 2D column vector (N, 1)
        self.y = torch.tensor(y_df.values, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        # Returns total number of rows
        return len(self.X)

    def __getitem__(self, idx):
        # Returns feature vector and target scalar for a given row index
        return self.X[idx], self.y[idx]

In [9]:
from pathlib import Path

split_dir = Path("../data/processed/eu/splits/final")

X_train_df = pd.read_parquet(split_dir / "X_train_encoded_scaled_final.parquet")
y_train_df = pd.read_parquet(split_dir / "y_train.parquet")

X_val_df = pd.read_parquet(split_dir / "X_val_encoded_scaled_final.parquet")
y_val_df = pd.read_parquet(split_dir / "y_val.parquet")

X_test_df = pd.read_parquet(split_dir / "X_test_encoded_scaled_final.parquet")
y_test_df = pd.read_parquet(split_dir / "y_test.parquet")

# Build DataLoaders
train_dataset = VehicleDataset(X_train_df, y_train_df)
val_dataset = VehicleDataset(X_val_df, y_val_df)
test_dataset = VehicleDataset(X_test_df, y_test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# --- Improved Architecture ---
#This is a 4-layer Multi-Layer Perceptron (MLP) built with nn.Sequential
class OptimizedMLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), #Linear(17 -> 128) projects 17 inputs into 128 dimensions
            nn.BatchNorm1d(128),#Normalizes activations within each mini-batch to speed up convergence and stabilize learning.
            nn.GELU(),#Gaussian Error Linear Unit provides smooth non-linear transitions
            nn.Dropout(0.05),#Randomly zeroes out 5% of hidden units during training to prevent over-reliance on specific nodes

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.05),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

# --- PyTorch Lightning Module ---
#LightningModule organizes model behavior across life cycle hooks
class OptimizedVehicleModel(LightningModule):
    def __init__(self, input_dim: int, lr: float = 1e-3, weight_decay: float = 1e-5):
        super().__init__()
        self.save_hyperparameters()
        self.network = OptimizedMLP(input_dim)
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        return self.network(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        preds = self(x)
        loss = self.loss_fn(preds, y.view_as(preds))
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        preds = self(x)
        y = y.view_as(preds)
        
        loss = self.loss_fn(preds, y)
        rmse = torch.sqrt(loss)
        
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_rmse", rmse, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        preds = self(x)
        y = y.view_as(preds)

        mse = self.loss_fn(preds, y)
        rmse = torch.sqrt(mse)
        mae = torch.abs(preds - y).mean()

        self.log("test_rmse", rmse, on_step=False, on_epoch=True)
        self.log("test_mae", mae, on_step=False, on_epoch=True)
        return {"test_rmse": rmse, "test_mae": mae}

    def configure_optimizers(self):
        #Adam with weight decay for L2 regularization
        optimizer = torch.optim.AdamW(
            self.parameters(), 
            lr=self.hparams.lr, 
            weight_decay=self.hparams.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=5
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
            },
        }

# --- Training Setup ---
# Input dimension based on the 17 selected features
input_dim = len(SELECTED_FEATURES)

model = OptimizedVehicleModel(input_dim=input_dim, lr=1e-3, weight_decay=1e-5)

# Callbacks for Early Stopping and Model Checkpointing
#Monitors val_loss and cancels training early if validation performance halts for 15 consecutive epochs.
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=15,
    mode="min",
    verbose=True
)

#Saves only the weights that achieve the lowest val_loss to a file (best_vehicle_model.ckpt)
checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    filename="best_vehicle_model",
    save_top_k=1,
    mode="min"
)

trainer = Trainer(
    max_epochs=100,
    callbacks=[early_stop_callback, checkpoint_callback],
    enable_progress_bar=True
)

# Train & Test
#Automatically reloads the best performing model weights from disk and evaluates on the test set.
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(ckpt_path="best", dataloaders=test_loader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network │ OptimizedMLP │ 13.2 K │ train │     0 │
│ 1 │ loss_fn │ MSELoss      │      0 │ train │     0 │
└───┴─────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 13.2 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 13.2 K                                                                                               
Total estimated model params size (MB): 0.053                                                                      
Modules in train mode: 15                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/Users/serhatgemici/Documents/Liora/jul26_bds_int_co2/.venv/lib/python3.13/site-packages/pytorch_lightning/utilitie
s/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and 
treespec.is_leaf()` instead.

...

Restoring states from the checkpoint path at /Users/serhatgemici/Documents/Liora/jul26_bds_int_co2/notebooks/lightning_logs/version_7/checkpoints/best_vehicle_model.ckpt
Loaded model weights from the checkpoint at /Users/serhatgemici/Documents/Liora/jul26_bds_int_co2/notebooks/lightning_logs/version_7/checkpoints/best_vehicle_model.ckpt


Output()

/Users/serhatgemici/Documents/Liora/jul26_bds_int_co2/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_mae          │     8.073248863220215     │
│         test_rmse         │    11.900761604309082     │
└───────────────────────────┴───────────────────────────┘

[{'test_rmse': 11.900761604309082, 'test_mae': 8.073248863220215}]